# Module 4: Search in Azure DocumentDB

**Time**: ~75 min  
**Environment**: Jupyter notebook in VS Code

This notebook is fully runnable. Enter your Azure DocumentDB connection string in Step 0, then run each cell in order. You will load sample documents, create vector and full-text indexes, run vector search, BM25 keyword search, fuzzy search, phrase search, and combine keyword + vector results with Reciprocal Rank Fusion (RRF).

> Full-text search in Azure DocumentDB is currently in gated preview. Vector DiskANN requires an M30 or higher cluster tier.


## Step 0: Connect to Azure DocumentDB

This cell restores the MongoDB driver, accepts your connection string, and connects to `docdbworkshop.workshop_content`.

In [ ]:
#r "nuget: MongoDB.Driver, 3.4.0"
using MongoDB.Bson;
using MongoDB.Driver;
using System.Linq;

var connectionString = Environment.GetEnvironmentVariable("DOCUMENTDB_CONNECTION_STRING") ?? "<paste-your-azure-documentdb-connection-string-here>";
if (connectionString.Contains("<paste")) throw new Exception("Paste your Azure DocumentDB connection string in this cell or set DOCUMENTDB_CONNECTION_STRING.");
var client = new MongoClient(connectionString);
var db = client.GetDatabase("docdbworkshop");
var collection = db.GetCollection<BsonDocument>("workshop_content");
db.RunCommand<BsonDocument>(new BsonDocument("ping", 1))

## Step 1: Load sample documents

The sample records include text for full-text search and simple vectors for semantic search.

In [ ]:
collection.DeleteMany(FilterDefinition<BsonDocument>.Empty);
collection.InsertMany(new[] {
    new BsonDocument { {"_id","doc-search-001"}, {"title","DiskANN vector indexing"}, {"category","vector"}, {"body","Azure DocumentDB supports DiskANN vector indexes for high recall semantic similarity search over embeddings stored with documents."}, {"sku","SEARCH-VEC-001"}, {"embedding",new BsonArray{0.92,0.80,0.18}} },
    new BsonDocument { {"_id","doc-search-002"}, {"title","BM25 keyword search"}, {"category","full-text"}, {"body","Azure DocumentDB full-text search ranks keyword matches with BM25 and exposes scores through searchScore metadata."}, {"sku","SEARCH-FTS-001"}, {"embedding",new BsonArray{0.20,0.12,0.94}} },
    new BsonDocument { {"_id","doc-search-003"}, {"title","Hybrid search with RRF"}, {"category","hybrid"}, {"body","Hybrid search combines BM25 keyword results with vector results and fuses the ranked lists using Reciprocal Rank Fusion."}, {"sku","SEARCH-HYB-001"}, {"embedding",new BsonArray{0.76,0.70,0.42}} },
    new BsonDocument { {"_id","doc-search-004"}, {"title","RAG grounding"}, {"category","rag"}, {"body","Retrieval augmented generation retrieves relevant chunks from Azure DocumentDB and grounds the model answer in that context."}, {"sku","RAG-PIPE-001"}, {"embedding",new BsonArray{0.82,0.74,0.36}} }
});
collection.CountDocuments(FilterDefinition<BsonDocument>.Empty)

## Step 2: Create indexes

Create one DiskANN vector index and one BM25 search index on the same collection.

In [ ]:
db.RunCommand<BsonDocument>(new BsonDocument{{"createIndexes","workshop_content"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_embedding_diskann"},{"key",new BsonDocument("embedding","cosmosSearch")},{"cosmosSearchOptions",new BsonDocument{{"kind","vector-diskann"},{"dimensions",3},{"similarity","COS"},{"maxDegree",32},{"lBuild",64}}}}}}});
db.RunCommand<BsonDocument>(new BsonDocument{{"createSearchIndexes","workshop_content"},{"indexes",new BsonArray{new BsonDocument{{"name","idx_body_fts"},{"definition",new BsonDocument("mappings",new BsonDocument{{"dynamic",false},{"fields",new BsonDocument("body",new BsonDocument("type","string"))}})}}}}});

## Step 3: Vector search

This aggregation uses `$search.cosmosSearch` to return the closest vectors.

In [ ]:
var vector = new BsonArray {0.90, 0.78, 0.22};
collection.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument("cosmosSearch", new BsonDocument{{"path","embedding"},{"vector",vector},{"k",3}})),
    new BsonDocument("$project", new BsonDocument{{"_id",0},{"title",1},{"category",1},{"score",new BsonDocument("$meta","searchScore")}})
}).ToList()

## Step 4: Full-text search

This aggregation uses BM25 keyword ranking through `$search.text`.

In [ ]:
collection.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query","BM25 ranking"},{"path","body"}}}}),
    new BsonDocument("$limit", 5),
    new BsonDocument("$project", new BsonDocument{{"_id",0},{"title",1},{"score",new BsonDocument("$meta","searchScore")}})
}).ToList()

## Step 5: Hybrid search with RRF

Run keyword and vector searches, then combine their ranks with Reciprocal Rank Fusion.

In [ ]:
double RrfScore(int rank, int k = 60) => 1.0 / (k + rank + 1);
var userQuery = "semantic retrieval for rag";
var queryVector = new BsonArray {0.84, 0.76, 0.32};
var keywordHits = collection.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument{{"index","idx_body_fts"},{"text",new BsonDocument{{"query",userQuery},{"path","body"}}}}),
    new BsonDocument("$limit", 5),
    new BsonDocument("$project", new BsonDocument{{"_id",1},{"title",1}})
}).ToList();
var vectorHits = collection.Aggregate<BsonDocument>(new[] {
    new BsonDocument("$search", new BsonDocument("cosmosSearch", new BsonDocument{{"path","embedding"},{"vector",queryVector},{"k",5}})),
    new BsonDocument("$project", new BsonDocument{{"_id",1},{"title",1}})
}).ToList();
var scores = new Dictionary<string,double>();
var titles = new Dictionary<string,string>();
foreach (var list in new[] { keywordHits, vectorHits })
{
    for (var i = 0; i < list.Count; i++)
    {
        var id = list[i]["_id"].ToString();
        titles[id] = list[i]["title"].ToString();
        scores[id] = scores.GetValueOrDefault(id) + RrfScore(i);
    }
}
scores.OrderByDescending(x => x.Value).Take(5).Select(x => new { Id = x.Key, Title = titles[x.Key], Score = x.Value }).ToList()